# 14 — Origin Classification: 4-Way Regional, Text + Numeric Scores Fusion

Multi-modal extension of notebook 11. Notebook 11 used scrubbed+ review text only and reached RoBERTa macro-F1 = 0.817 (accuracy 0.834). Coffee Review provides per-review sensory scores (Rating, Aroma, Acidity, Body, Flavor, Aftertaste) which are systematically different across regions (East African coffees score higher on Acidity, Indonesian coffees on Body, etc). This notebook fuses them with the RoBERTa [CLS] embedding before classification.

**Architecture (fusion model):**
```
  text  -> RoBERTa  -> [CLS] (768d)  ----+
                                          \\
  scores (6d) -> MLP (-> 64d) -----------> [concat 832d] -> MLP -> 4 region logits
```

**Score features used** (6 total):
- `Rating` (overall, 63-98) — 0% missing
- `Aroma` (4-10) — 1.2% missing
- `Combined_Acidity` (2-10, merges Acidity + Acidity/Structure) — 13.1% missing
- `Body` (5-10) — 0.7% missing
- `Flavor` (3-10) — 0.7% missing
- `Aftertaste` (2-10) — 4.1% missing

Missing values are imputed with train-set mean. Features are z-scored using train-set mean/std.

**Experiments:**
- **A.** Scores-only sklearn LogisticRegression baseline — establishes how much signal is in numeric scores alone.
- **B.** Text-only RoBERTa — *reference is notebook 11* (RoBERTa 0.817 ± 0.007). No retrain.
- **C.** Text + scores fusion (RoBERTa + score MLP), weighted CE × 3 seeds — main result.

Same train/val/test split scheme as 11 (stratified by region, 70/15/15).


In [1]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    balanced_accuracy_score, f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModel, AutoTokenizer, DataCollatorWithPadding,
    EarlyStoppingCallback, Trainer, TrainingArguments,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.utils.notebook import NotebookProgressCallback

RANDOM_STATE = 42
SEEDS = [42, 123, 2024]
TEXT_COLUMN = 'text_full_concat_scrubbed_plus'
NUMERIC_COLUMNS = ['Rating', 'Aroma', 'Combined_Acidity', 'Body', 'Flavor', 'Aftertaste']
MAX_LENGTH = 256

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
USE_FP16 = True

ROBERTA_CKPT = 'roberta-base'
ROBERTA_LR = 2e-5
SCORE_PROJ_DIM = 64
FUSION_HIDDEN_DIM = 256
DROPOUT = 0.1

OUTPUT_DIR_ROOT = 'artifacts/origin_region_4way_text_plus_scores'
os.makedirs(OUTPUT_DIR_ROOT, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

COUNTRY_TO_REGION = {
    # --- East Africa ---
    'Ethiopia': 'East Africa',
    'Kenya': 'East Africa',
    'Rwanda': 'East Africa',
    'Burundi': 'East Africa',
    'Tanzania': 'East Africa',
    'Uganda': 'East Africa',
    'DR Congo': 'East Africa',
    'Zambia': 'East Africa',
    'Zimbabwe': 'East Africa',
    'Cameroon': 'East Africa',
    'Malawi': 'East Africa',
    'South Africa': 'East Africa',
    'Yemen': 'East Africa',
    # --- Central America ---
    'Guatemala': 'Central America',
    'Costa Rica': 'Central America',
    'Panama': 'Central America',
    'El Salvador': 'Central America',
    'Honduras': 'Central America',
    'Nicaragua': 'Central America',
    'Mexico': 'Central America',
    'Jamaica': 'Central America',
    'Puerto Rico': 'Central America',
    'Haiti': 'Central America',
    'Dominican Republic': 'Central America',
    # --- South America ---
    'Colombia': 'South America',
    'Peru': 'South America',
    'Brazil': 'South America',
    'Ecuador': 'South America',
    'Bolivia': 'South America',
    'Venezuela': 'South America',
    # --- Asia-Pacific ---
    'Indonesia': 'Asia-Pacific',
    'Taiwan': 'Asia-Pacific',
    'Thailand': 'Asia-Pacific',
    'Papua New Guinea': 'Asia-Pacific',
    'Philippines': 'Asia-Pacific',
    'India': 'Asia-Pacific',
    'Vietnam': 'Asia-Pacific',
    'China': 'Asia-Pacific',
    'Timor-Leste': 'Asia-Pacific',
    'Malaysia': 'Asia-Pacific',
    'Laos': 'Asia-Pacific',
    'Nepal': 'Asia-Pacific',
    'Myanmar': 'Asia-Pacific',
    'Australia': 'Asia-Pacific',
    'United Kingdom': 'Asia-Pacific',
    'United States': 'Asia-Pacific',
}


Device: cuda


## Scrubbing vocabulary (identical to 07.1 / 04.5 / 11)


In [2]:
# Scrubbing vocabulary: countries + region aliases + cultivars + producer context
COUNTRY_TERMS = {
    "Ethiopia": ["ethiopia", "ethiopian"],
    "Colombia": ["colombia", "colombian", "columbian"],
    "Panama": ["panama", "panamanian"],
    "Kenya": ["kenya", "kenyan"],
    "Indonesia": ["indonesia", "indonesian"],
    "Guatemala": ["guatemala", "guatemalan"],
    "Costa Rica": ["costa rica", "costa rican", "costarican"],
    "El Salvador": ["el salvador", "salvadoran", "salvadorean", "salvadorian"],
    "Rwanda": ["rwanda", "rwandan"],
    "Brazil": ["brazil", "brazilian"],
    "Honduras": ["honduras", "honduran"],
    "Peru": ["peru", "peruvian"],
    "Taiwan": ["taiwan", "taiwanese"],
    "Papua New Guinea": ["papua new guinea", "papua new guinean"],
    "Nicaragua": ["nicaragua", "nicaraguan"],
    "Burundi": ["burundi", "burundian"],
    "Thailand": ["thailand", "thai"],
    "India": ["india", "indian"],
    "Mexico": ["mexico", "mexican"],
    "Tanzania": ["tanzania", "tanzanian"],
    "Yemen": ["yemen", "yemeni"],
    "Ecuador": ["ecuador", "ecuadorian"],
    "Bolivia": ["bolivia", "bolivian"],
    "Jamaica": ["jamaica", "jamaican"],
    "Uganda": ["uganda", "ugandan"],
    "Dominican Republic": ["dominican republic", "dominican"],
    "Zambia": ["zambia", "zambian"],
    "China": ["china", "chinese"],
    "Vietnam": ["vietnam", "vietnamese"],
    "Philippines": ["philippines", "philippine", "filipino"],
    "Malaysia": ["malaysia", "malaysian"],
    "Laos": ["laos", "laotian"],
    "Zimbabwe": ["zimbabwe", "zimbabwean"],
    "Haiti": ["haiti", "haitian"],
    "Puerto Rico": ["puerto rico", "puerto rican", "puertorican"],
    "Nepal": ["nepal", "nepalese", "nepali"],
    "Myanmar": ["myanmar", "burmese"],
    "Cameroon": ["cameroon", "cameroonian"],
    "Australia": ["australia", "australian"],
    "South Africa": ["south africa", "south african"],
    "Malawi": ["malawi", "malawian"],
    "Venezuela": ["venezuela", "venezuelan", "merida state", "mocoties valley"],
    "Timor-Leste": ["east timor", "timor leste", "timorese"],
    "DR Congo": ["democratic republic of the congo", "dr congo", "congo", "congolese"],
    "United Kingdom": ["united kingdom", "british", "pitcairn island", "saint helena", "st helena", "sandy bay valley"],
    "United States": [
        "usa", "united states", "american",
        "hawaii", "hawai'i", "hawaiian", "hawaii island",
        "big island", "kona", "puna district", "holualoa", "oahu", "maui", "kauai",
        "ka u", "ka'u", "kau",
    ],
}

REGION_ALIASES = {
    "Ethiopia": ["yirgacheffe", "sidamo", "sidama", "guji", "gedeb", "gedeo", "kochere", "hambela",
                 "shakiso", "jimma", "limu", "oromia", "harrar", "kaffa", "bench maji", "bench-maji",
                 "arbegona", "bensa"],
    "Colombia": ["huila", "cauca", "narino", "tolima", "quindio", "caldas", "risaralda", "antioquia",
                 "cundinamarca", "santander", "pitalito", "acevedo", "planadas", "gaitania", "piendamo",
                 "caicedonia", "san agustin", "armenia"],
    "Panama": ["boquete", "chiriqui", "volcan", "jaramillo", "alto quiel", "paso ancho",
               "piedra candela", "canas verdes", "silla del pando", "renacimiento"],
    "Kenya": ["nyeri", "kirinyaga", "kiambu", "embu", "muranga", "murang'a", "thika", "ruiru",
              "mathira", "meru", "nakuru", "gichugu", "karatina"],
    "Indonesia": ["sumatra", "aceh", "gayo", "lintong", "mandheling", "sidikalang", "toraja",
                  "sulawesi", "java", "bali", "kintamani", "flores", "kerinci"],
    "Guatemala": ["huehuetenango", "antigua", "acatenango", "fraijanes", "coban", "atitlan",
                  "chimaltenango", "quiche", "solola", "palencia", "sacatepequez", "san marcos",
                  "hoja blanca", "cuilco"],
    "Costa Rica": ["tarrazu", "central valley", "west valley", "tres rios", "naranjo", "dota",
                   "poas", "alajuela", "brunca", "turrialba", "coto brus", "chirripo"],
    "El Salvador": ["ahuachapan", "chalatenango", "apaneca", "ilamatepec", "santa ana",
                    "el boqueron", "quetzaltepec", "juayua", "ataco"],
    "Rwanda": ["gakenke", "nyamasheke", "karongi", "huye", "nyamagabe", "rulindo", "gikongoro",
               "lake kivu"],
    "Brazil": ["cerrado", "mogiana", "minas gerais", "mantiqueira", "sul de minas",
               "chapada diamantina", "carmo de minas"],
    "Honduras": ["marcala", "copan", "ocotepeque", "comayagua", "intibuca", "santa barbara",
                 "el paraiso", "capucas"],
    "Peru": ["cajamarca", "jaen", "san ignacio", "chanchamayo", "cusco", "villa rica", "oxapampa",
             "junin", "puno"],
    "Taiwan": ["alishan", "yunlin", "chiayi", "chia yi", "nantou", "taichung", "pingtung"],
    "Papua New Guinea": ["wahgi valley", "western highlands", "eastern highlands", "jiwaka",
                         "kainantu", "okapa"],
    "Nicaragua": ["jinotega", "matagalpa", "nueva segovia", "madriz", "ocotal", "dipilto"],
    "Burundi": ["kayanza", "ngozi", "muramvya", "muyinga", "bururi"],
    "Thailand": ["chiang rai", "doi chang", "doi pangkhon", "chiang mai", "nan province"],
    "India": ["coorg", "chikmagalur", "karnataka", "bababudangiri", "nilgiris"],
    "Mexico": ["chiapas", "oaxaca", "veracruz", "coatepec", "pluma hidalgo"],
    "Tanzania": ["mbeya", "ruvuma", "ngorongoro", "arusha", "kilimanjaro", "mbozi"],
    "Yemen": ["haraaz", "haraz", "sanaa", "bani matar", "hayma"],
    "Ecuador": ["loja", "pichincha", "imbabura", "zamora", "chimborazo", "saraguro"],
    "Bolivia": ["caranavi", "yungas", "la paz"],
    "Jamaica": ["blue mountain", "blue mountains"],
    "Uganda": ["rwenzori", "bugisu", "sipi falls"],
    "Vietnam": ["lam dong", "quang tri", "dalat", "cau dat"],
    "Philippines": ["benguet", "bukidnon", "davao"],
    "China": ["yunnan", "baoshan", "puer", "pu'er", "lincong"],
    "Puerto Rico": ["utuado", "yauco", "adjuntas"],
    "Venezuela": ["mocoties valley", "merida state"],
    "Malawi": ["malawi"],
}

REGION_ONLY_TERMS = [
    "central america", "south america", "latin america",
    "east africa", "central africa", "west africa", "africa",
    "asia", "the americas", "americas",
    "central and south america", "south and central america",
    "east and central africa", "various africa growing regions",
]

CULTIVAR_TERMS = [
    "sl28", "sl-28", "sl 28", "sl34", "sl-34", "sl 34",
    "ruiru 11", "ruiru-11", "batian", "k7",
    "gesha", "geisha",
    "pacamara", "pache", "villa sarchi",
    "bourbon", "pink bourbon", "yellow bourbon", "red bourbon",
    "caturra", "catuai", "catuai vermelho", "catuai amarelo",
    "typica", "mundo novo",
    "maragogipe", "maragogype", "maracaturra",
    "castillo", "colombia variety", "variedad colombia", "tabi",
    "heirloom", "ethiopian heirloom",
    "tim tim", "s795", "ateng",
    "catimor", "sarchimor", "icatu", "obata",
]

PRODUCER_CONTEXT_TERMS = [
    "finca", "hacienda", "beneficio",
    "cup of excellence",
    "washing station", "wet mill",
    "best of panama",
]

all_scrub_terms = []
for v in COUNTRY_TERMS.values():    all_scrub_terms.extend(v)
for v in REGION_ALIASES.values():   all_scrub_terms.extend(v)
all_scrub_terms.extend(REGION_ONLY_TERMS)
all_scrub_terms.extend(CULTIVAR_TERMS)
all_scrub_terms.extend(PRODUCER_CONTEXT_TERMS)
all_scrub_terms = sorted(set(all_scrub_terms), key=len, reverse=True)
print(f'Loaded {len(all_scrub_terms)} scrub terms')

SCRUB_PATTERN = re.compile(
    r'\b(' + '|'.join(re.escape(t) for t in all_scrub_terms) + r')\b',
    flags=re.IGNORECASE,
)

def scrub(text):
    if not isinstance(text, str) or not text:
        return ''
    return SCRUB_PATTERN.sub(' [ORIGIN] ', text)


Loaded 403 scrub terms


## Build text + numeric features; map regions; filter


In [3]:
def minimal_raw_text(text):
    text = '' if pd.isna(text) else str(text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def normalize_text_keep_brackets(text):
    text = minimal_raw_text(text)
    text = re.sub(r'[^a-z0-9\s\[\]]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

EXTRA_TEXT_COLS = ['Blind Assessment', 'Notes', 'Who Should Drink It', 'Bottom Line']

def concat_and_scrub(row):
    pieces = []
    for col in EXTRA_TEXT_COLS:
        v = row[col]
        if pd.notna(v) and str(v).strip():
            raw = minimal_raw_text(v)
            scrubbed = scrub(raw)
            cleaned = normalize_text_keep_brackets(scrubbed)
            pieces.append(cleaned)
    return ' '.join(p for p in pieces if p)

df = pd.read_csv('Data/final_coffee_reviews.csv')
df['text_full_concat_scrubbed_plus'] = df.apply(concat_and_scrub, axis=1)
df['text_raw_minimal'] = df['Blind Assessment'].fillna('').map(minimal_raw_text)
df['origin_country'] = df['Country'].astype('string').str.strip().replace('', pd.NA)
df['origin_region']  = df['origin_country'].map(COUNTRY_TO_REGION)

work = df[
    (df['text_raw_minimal'].str.len() >= 30)
    & (df['origin_country'].notna())
    & (df['origin_region'].notna())
].copy().reset_index(drop=True)

print(f'Rows: {len(work)} | Regions: {work["origin_region"].nunique()}')
print()
print('Per-region missing rate by score column:')
for col in NUMERIC_COLUMNS:
    miss = work.groupby('origin_region')[col].apply(lambda s: s.isna().mean())
    print(f'  {col:18s}  {dict(miss.round(3))}')
print()
print('Per-region mean by score column (gives a sense of how discriminative the scores are):')
for col in NUMERIC_COLUMNS:
    means = work.groupby('origin_region')[col].mean().round(2)
    print(f'  {col:18s}  {dict(means)}')


Rows: 7585 | Regions: 4

Per-region missing rate by score column:
  Rating              {'Asia-Pacific': 0.0, 'Central America': 0.001, 'East Africa': 0.0, 'South America': 0.0}
  Aroma               {'Asia-Pacific': 0.014, 'Central America': 0.016, 'East Africa': 0.009, 'South America': 0.011}
  Combined_Acidity    {'Asia-Pacific': 0.135, 'Central America': 0.106, 'East Africa': 0.143, 'South America': 0.132}
  Body                {'Asia-Pacific': 0.014, 'Central America': 0.011, 'East Africa': 0.003, 'South America': 0.004}
  Flavor              {'Asia-Pacific': 0.014, 'Central America': 0.011, 'East Africa': 0.002, 'South America': 0.004}
  Aftertaste          {'Asia-Pacific': 0.091, 'Central America': 0.057, 'East Africa': 0.017, 'South America': 0.035}

Per-region mean by score column (gives a sense of how discriminative the scores are):
  Rating              {'Asia-Pacific': 91.4, 'Central America': 91.96, 'East Africa': 92.83, 'South America': 91.93}
  Aroma               {'Asia

## Top-level stratified split (by region, 70/15/15) — shared across all experiments


In [4]:
y_region = work['origin_region']
idx = np.arange(len(work))

idx_tmp, idx_test = train_test_split(
    idx, test_size=0.15, stratify=y_region, random_state=RANDOM_STATE,
)
idx_train, idx_val = train_test_split(
    idx_tmp, test_size=0.15/0.85,
    stratify=y_region.iloc[idx_tmp], random_state=RANDOM_STATE,
)

train_df = work.iloc[idx_train].copy().reset_index(drop=True)
val_df   = work.iloc[idx_val].copy().reset_index(drop=True)
test_df  = work.iloc[idx_test].copy().reset_index(drop=True)

label_names = sorted(work['origin_region'].unique().tolist())
label2id = {n: i for i, n in enumerate(label_names)}
id2label = {i: n for i, n in enumerate(label_names)}

# Numeric features: impute missing with TRAIN-set mean, then z-score with TRAIN-set mean/std
train_means = train_df[NUMERIC_COLUMNS].mean()
train_stds  = train_df[NUMERIC_COLUMNS].std().replace(0, 1.0)

def featurize_scores(part_df):
    X = part_df[NUMERIC_COLUMNS].copy()
    for col in NUMERIC_COLUMNS:
        X[col] = X[col].fillna(train_means[col])
    X = (X - train_means) / train_stds
    return X.to_numpy(dtype=np.float32)

X_scores_train = featurize_scores(train_df)
X_scores_val   = featurize_scores(val_df)
X_scores_test  = featurize_scores(test_df)

y_train = np.array([label2id[r] for r in train_df['origin_region']])
y_val   = np.array([label2id[r] for r in val_df['origin_region']])
y_test  = np.array([label2id[r] for r in test_df['origin_region']])

class_weights_np = compute_class_weight(class_weight='balanced',
                                        classes=np.arange(len(label_names)), y=y_train)
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float32)

print(f'Train/Val/Test: {len(train_df)} / {len(val_df)} / {len(test_df)}')
print(f'Labels: {label_names}')
print(f'Class weights: {dict(zip(label_names, class_weights_np.round(3)))}')
print(f'Score-feature shape (train): {X_scores_train.shape}')
print(f'Train means used for impute/scale: {train_means.round(2).to_dict()}')


Train/Val/Test: 5309 / 1138 / 1138
Labels: ['Asia-Pacific', 'Central America', 'East Africa', 'South America']
Class weights: {'Asia-Pacific': 1.784, 'Central America': 0.98, 'East Africa': 0.604, 'South America': 1.313}
Score-feature shape (train): (5309, 6)
Train means used for impute/scale: {'Rating': 92.25, 'Aroma': 8.64, 'Combined_Acidity': 8.29, 'Body': 8.37, 'Flavor': 8.76, 'Aftertaste': 8.07}


## Experiment A — Scores-only baseline (sklearn LogisticRegression)


In [5]:
# Sanity floor: how much region signal is in the numeric scores alone?
clf = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)
clf.fit(X_scores_train, y_train)

scores_only_results = {}
for split_name, X_split, y_split in [
    ('val',  X_scores_val,  y_val),
    ('test', X_scores_test, y_test),
]:
    pred = clf.predict(X_split)
    prec, rec, f1, _ = precision_recall_fscore_support(y_split, pred, average='macro', zero_division=0)
    scores_only_results[split_name] = {
        'f1_macro': float(f1),
        'bal_acc': float(balanced_accuracy_score(y_split, pred)),
        'accuracy': float(accuracy_score(y_split, pred)),
    }

print('Scores-only LogisticRegression (4-way region):')
for split, m in scores_only_results.items():
    print(f'  {split:5s}  F1={m["f1_macro"]:.4f}  bal-acc={m["bal_acc"]:.4f}  acc={m["accuracy"]:.4f}')

print()
print('Per-class precision / recall / F1 on test:')
pred_test = clf.predict(X_scores_test)
prec_c, rec_c, f1_c, sup_c = precision_recall_fscore_support(y_test, pred_test, labels=range(len(label_names)), zero_division=0)
for i, name in enumerate(label_names):
    print(f'  {name:18s}  P={prec_c[i]:.3f}  R={rec_c[i]:.3f}  F1={f1_c[i]:.3f}  (n={sup_c[i]})')


Scores-only LogisticRegression (4-way region):
  val    F1=0.2840  bal-acc=0.2930  acc=0.3401
  test   F1=0.3094  bal-acc=0.3198  acc=0.3664

Per-class precision / recall / F1 on test:
  Asia-Pacific        P=0.256  R=0.319  F1=0.284  (n=160)
  Central America     P=0.267  R=0.348  F1=0.302  (n=290)
  East Africa         P=0.525  R=0.520  F1=0.522  (n=471)
  South America       P=0.213  R=0.092  F1=0.129  (n=217)


## Experiment C — Text + Scores Fusion (RoBERTa + score MLP) × 3 seeds


In [6]:
class CoffeeFusionDataset(torch.utils.data.Dataset):
    def __init__(self, texts, scores, labels, tokenizer, max_length):
        self.encodings = tokenizer(texts, truncation=True, max_length=max_length, padding=False)
        self.scores = scores
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['scores'] = torch.tensor(self.scores[idx], dtype=torch.float32)
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class FusionCollator:
    """Pads input_ids/attention_mask normally, stacks scores tensors separately."""
    def __init__(self, tokenizer):
        self.padder = DataCollatorWithPadding(tokenizer=tokenizer)
    def __call__(self, features):
        scores = torch.stack([f['scores'] for f in features])
        labels = torch.stack([f['labels'] for f in features])
        rest = [{k: v for k, v in f.items() if k not in ('scores', 'labels')} for f in features]
        batch = self.padder(rest)
        batch['scores'] = scores
        batch['labels'] = labels
        return batch

class RobertaWithScores(nn.Module):
    """RoBERTa [CLS] hidden state concatenated with a small MLP-projected score vector."""
    def __init__(self, model_name, num_labels, num_score_features,
                 score_proj_dim=SCORE_PROJ_DIM, hidden_dim=FUSION_HIDDEN_DIM,
                 dropout=DROPOUT, class_weights=None):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        enc_hidden = self.encoder.config.hidden_size
        self.score_proj = nn.Sequential(
            nn.Linear(num_score_features, score_proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Sequential(
            nn.Linear(enc_hidden + score_proj_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_labels),
        )
        self.num_labels = num_labels
        self.class_weights = class_weights
        # Trainer expects this for some bookkeeping
        self.config = self.encoder.config
        self.config.num_labels = num_labels

    def forward(self, input_ids=None, attention_mask=None, scores=None,
                labels=None, **kwargs):
        enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = enc_out.last_hidden_state[:, 0, :]
        score_repr = self.score_proj(scores)
        fused = torch.cat([cls, score_repr], dim=-1)
        logits = self.classifier(fused)
        loss = None
        if labels is not None:
            if self.class_weights is not None:
                cw = self.class_weights.to(logits.device)
                loss_fct = nn.CrossEntropyLoss(weight=cw)
            else:
                loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)
        return SequenceClassifierOutput(loss=loss, logits=logits)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {
        'accuracy': accuracy_score(labels, preds),
        'balanced_accuracy': balanced_accuracy_score(labels, preds),
        'precision_macro': precision, 'recall_macro': recall, 'f1_macro': f1,
    }


def run_fusion(seed, tag):
    set_seed(seed)
    out_dir = os.path.join(OUTPUT_DIR_ROOT, tag)

    tokenizer = AutoTokenizer.from_pretrained(ROBERTA_CKPT)
    train_ds = CoffeeFusionDataset(train_df[TEXT_COLUMN].tolist(), X_scores_train, y_train.tolist(), tokenizer, MAX_LENGTH)
    val_ds   = CoffeeFusionDataset(val_df[TEXT_COLUMN].tolist(),   X_scores_val,   y_val.tolist(),   tokenizer, MAX_LENGTH)
    test_ds  = CoffeeFusionDataset(test_df[TEXT_COLUMN].tolist(),  X_scores_test,  y_test.tolist(),  tokenizer, MAX_LENGTH)
    collator = FusionCollator(tokenizer)

    model = RobertaWithScores(
        ROBERTA_CKPT,
        num_labels=len(label_names),
        num_score_features=X_scores_train.shape[1],
        class_weights=class_weights_tensor,
    )

    args = TrainingArguments(
        output_dir=out_dir, learning_rate=ROBERTA_LR,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=12, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        eval_strategy='epoch', logging_strategy='epoch',
        save_strategy='epoch', save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro', greater_is_better=True,
        disable_tqdm=True, report_to='none',
        fp16=USE_FP16 and torch.cuda.is_available(), seed=seed,
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        processing_class=tokenizer, data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )
    try: trainer.remove_callback(NotebookProgressCallback)
    except Exception: pass

    print(f'\n=== {tag} | seed={seed} ===')
    trainer.train()
    val_m  = trainer.evaluate(eval_dataset=val_ds)
    test_m = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix='test')

    del model, trainer
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return {
        'tag': tag, 'seed': seed,
        'val_f1_macro':  val_m['eval_f1_macro'],
        'val_bal_acc':   val_m['eval_balanced_accuracy'],
        'val_accuracy':  val_m['eval_accuracy'],
        'test_f1_macro': test_m['test_f1_macro'],
        'test_bal_acc':  test_m['test_balanced_accuracy'],
        'test_accuracy': test_m['test_accuracy'],
    }


fusion_results = []
for s in SEEDS:
    fusion_results.append(run_fusion(seed=s, tag=f'fusion_seed{s}'))

fusion_df = pd.DataFrame(fusion_results)[['seed','val_f1_macro','test_f1_macro','test_bal_acc','test_accuracy']]
print()
print('Fusion (text + scores) per-seed:')
print(fusion_df.round(4).to_string(index=False))
print()
print('Fusion mean ± std:')
print(fusion_df.drop(columns=['seed']).agg(['mean','std']).round(4).to_string())


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



=== fusion_seed42 | seed=42 ===
{'loss': '2.664', 'grad_norm': '22.96', 'learning_rate': '1.955e-05', 'epoch': '1'}
{'eval_loss': '1.1', 'eval_accuracy': '0.5791', 'eval_balanced_accuracy': '0.4627', 'eval_precision_macro': '0.3982', 'eval_recall_macro': '0.4627', 'eval_f1_macro': '0.4106', 'eval_runtime': '1.722', 'eval_samples_per_second': '660.7', 'eval_steps_per_second': '20.9', 'epoch': '1'}
{'loss': '1.922', 'grad_norm': '25.21', 'learning_rate': '1.779e-05', 'epoch': '2'}
{'eval_loss': '0.8135', 'eval_accuracy': '0.7153', 'eval_balanced_accuracy': '0.664', 'eval_precision_macro': '0.6956', 'eval_recall_macro': '0.664', 'eval_f1_macro': '0.6729', 'eval_runtime': '1.718', 'eval_samples_per_second': '662.5', 'eval_steps_per_second': '20.96', 'epoch': '2'}
{'loss': '1.402', 'grad_norm': '36.79', 'learning_rate': '1.601e-05', 'epoch': '3'}
{'eval_loss': '0.6589', 'eval_accuracy': '0.739', 'eval_balanced_accuracy': '0.7186', 'eval_precision_macro': '0.7553', 'eval_recall_macro': '0.7

early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'test_loss': '0.6632', 'test_accuracy': '0.8286', 'test_balanced_accuracy': '0.8065', 'test_precision_macro': '0.8182', 'test_recall_macro': '0.8065', 'test_f1_macro': '0.8106', 'test_runtime': '1.653', 'test_samples_per_second': '688.4', 'test_steps_per_second': '21.78', 'epoch': '12'}


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



=== fusion_seed2024 | seed=2024 ===
{'loss': '2.568', 'grad_norm': '17.09', 'learning_rate': '1.954e-05', 'epoch': '1'}
{'eval_loss': '0.936', 'eval_accuracy': '0.6731', 'eval_balanced_accuracy': '0.6082', 'eval_precision_macro': '0.6361', 'eval_recall_macro': '0.6082', 'eval_f1_macro': '0.6162', 'eval_runtime': '1.678', 'eval_samples_per_second': '678.1', 'eval_steps_per_second': '21.45', 'epoch': '1'}
{'loss': '1.643', 'grad_norm': '29.23', 'learning_rate': '1.777e-05', 'epoch': '2'}
{'eval_loss': '0.6959', 'eval_accuracy': '0.7285', 'eval_balanced_accuracy': '0.7088', 'eval_precision_macro': '0.7115', 'eval_recall_macro': '0.7088', 'eval_f1_macro': '0.6959', 'eval_runtime': '1.715', 'eval_samples_per_second': '663.6', 'eval_steps_per_second': '20.99', 'epoch': '2'}
{'loss': '1.178', 'grad_norm': '16.94', 'learning_rate': '1.6e-05', 'epoch': '3'}
{'eval_loss': '0.6421', 'eval_accuracy': '0.7408', 'eval_balanced_accuracy': '0.7287', 'eval_precision_macro': '0.7764', 'eval_recall_macr

early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'test_loss': '0.5726', 'test_accuracy': '0.8181', 'test_balanced_accuracy': '0.79', 'test_precision_macro': '0.8084', 'test_recall_macro': '0.79', 'test_f1_macro': '0.7961', 'test_runtime': '1.685', 'test_samples_per_second': '675.4', 'test_steps_per_second': '21.36', 'epoch': '8'}

Fusion (text + scores) per-seed:
 seed  val_f1_macro  test_f1_macro  test_bal_acc  test_accuracy
   42        0.8047         0.8106        0.8065         0.8286
  123        0.8077         0.8179        0.8149         0.8357
 2024        0.8023         0.7961        0.7900         0.8181

Fusion mean ± std:
      val_f1_macro  test_f1_macro  test_bal_acc  test_accuracy
mean        0.8049         0.8082        0.8038         0.8275
std         0.0027         0.0111        0.0127         0.0088


## Head-to-head summary


In [7]:
print('=' * 70)
print('4-way regional classification, scrubbed+ full-concat, 3-seed mean ± std')
print('=' * 70)
print()
print(f'  A. Scores-only (LogReg):      F1={scores_only_results["test"]["f1_macro"]:.4f}  '
      f'bal-acc={scores_only_results["test"]["bal_acc"]:.4f}  '
      f'acc={scores_only_results["test"]["accuracy"]:.4f}  (single fit, deterministic)')
print()
print(f'  B. Text-only RoBERTa (NB 11): F1=0.8171 ± 0.0071              acc=0.8345 ± 0.0048')
print()
print(f'  C. Text + Scores (this NB):   F1={fusion_df["test_f1_macro"].mean():.4f} ± {fusion_df["test_f1_macro"].std():.4f}  '
      f'bal-acc={fusion_df["test_bal_acc"].mean():.4f} ± {fusion_df["test_bal_acc"].std():.4f}  '
      f'acc={fusion_df["test_accuracy"].mean():.4f} ± {fusion_df["test_accuracy"].std():.4f}')
print()
delta_f1  = fusion_df["test_f1_macro"].mean() - 0.8171
delta_acc = fusion_df["test_accuracy"].mean() - 0.8345
print(f'  Fusion vs text-only (NB 11):  ΔF1 = {delta_f1:+.4f}   Δacc = {delta_acc:+.4f}')


4-way regional classification, scrubbed+ full-concat, 3-seed mean ± std

  A. Scores-only (LogReg):      F1=0.3094  bal-acc=0.3198  acc=0.3664  (single fit, deterministic)

  B. Text-only RoBERTa (NB 11): F1=0.8171 ± 0.0071              acc=0.8345 ± 0.0048

  C. Text + Scores (this NB):   F1=0.8082 ± 0.0111  bal-acc=0.8038 ± 0.0127  acc=0.8275 ± 0.0088

  Fusion vs text-only (NB 11):  ΔF1 = -0.0089   Δacc = -0.0070


## Save results


In [8]:
out = {
    'notebook': '14_Origin_Regional_4Way_TextPlusScores',
    'task': '4-way regional classification with text+scores fusion',
    'text_column': TEXT_COLUMN,
    'numeric_columns': NUMERIC_COLUMNS,
    'n_rows': int(len(work)),
    'n_classes': int(work['origin_region'].nunique()),
    'region_distribution': work['origin_region'].value_counts().to_dict(),
    'scores_only_baseline': scores_only_results,
    'fusion_runs': fusion_results,
    'fusion_summary': {
        'test_f1_macro_mean':  float(fusion_df['test_f1_macro'].mean()),
        'test_f1_macro_std':   float(fusion_df['test_f1_macro'].std()),
        'test_bal_acc_mean':   float(fusion_df['test_bal_acc'].mean()),
        'test_bal_acc_std':    float(fusion_df['test_bal_acc'].std()),
        'test_accuracy_mean':  float(fusion_df['test_accuracy'].mean()),
        'test_accuracy_std':   float(fusion_df['test_accuracy'].std()),
    },
    'reference_text_only_nb11': {
        'test_f1_macro_mean': 0.8171, 'test_f1_macro_std': 0.0071,
        'test_accuracy_mean': 0.8345, 'test_accuracy_std': 0.0048,
    },
}
out_path = os.path.join(OUTPUT_DIR_ROOT, 'results.json')
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2, default=float)
print('Saved:', out_path)


Saved: artifacts/origin_region_4way_text_plus_scores\results.json
